This notebook is designated for training, just copying and pasting from different scripts

In [ ]:
%pip install osfclient torchcodec

In [ ]:
import os
import zipfile
from osfclient import OSF

"""
Downloading the pushT data from the osf project for DINO-WM which the data was first
collected for but LeWM also used it so running this script accesses the project and
downloads and unzips the dataset
"""

PROJECT_ID = "bmw48"
TARGET_ZIP_NAME = "pusht_noise.zip"

osf = OSF()
project = osf.project(PROJECT_ID)
storage = project.storage('osfstorage')

download_dir = "data"
os.makedirs(download_dir, exist_ok=True)

found = False
for f in storage.files:
    if f.name != TARGET_ZIP_NAME:
        continue
    found = True

    out_path = os.path.join(download_dir, TARGET_ZIP_NAME)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    with open(out_path, "wb") as fp:
        f.write_to(fp)

    print(f"Downloaded: {storage.provider}:{f.path} -> {out_path}")

    extract_dir = os.path.join(download_dir, os.path.splitext(TARGET_ZIP_NAME)[0])
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(out_path, "r") as zf:
        zf.extractall(extract_dir)

    os.remove(out_path)
    print(f"Unzipped to: {extract_dir}")
    print(f"Deleted zip: {out_path}")

    break

if not found:
    raise SystemExit(f"Could not find {TARGET_ZIP_NAME!r} in OSF project {PROJECT_ID!r}.")

In [ ]:
import torch
from torch import Tensor
from torch.linalg import inv_ex
from torch.utils.data import Dataset
from pathlib import Path
import torchvision
from torchcodec.decoders import VideoDecoder

"""
This script is for processing the data from the raw pusht files:
- observations come in the form of videos and these have a max of 246 frames
    - sometimes the episode ends early and the rest of the actions are padded so they can be
	  a single tensor
- there are delta position based action tensors that correspond to each frame which will be used
- to make inference more efficient, frame skips are utilized so instead of predicting the next frame
  you predict the Nth next frame
- if frame skip was 2 this would make the data pairs:
  - x: (frame0, a0 + a1), y: frame3
"""

class PushTDataset(Dataset):
	def __init__(self,
				 data_dir : Path,
				 frame_skip : int,
				 window : int,
				 val : bool = False):
		super().__init__()
		dir_var = "val" if val else "train"

		self.frame_skip = frame_skip
		self.window = window
		self.action_tensor = torch.load(data_dir / "pusht_noise" / "pusht_noise" / dir_var / "rel_actions.pth")
		# shape of action tensor: [18685, 246, 2] : [episodes, action per frame + padding, ]
		obs_dir = Path(data_dir / "pusht_noise" / "pusht_noise" / dir_var / "obses")
		video_dirs = [ep for ep in obs_dir.iterdir()] # list of Path objects for each vid 
		# data will be returned like this frames : [window of inputs frames, next frame], [window of actions]
		# only valid batches of frames will be returned, not the 
		self.frame_samples = []
		for idx, ep in enumerate(video_dirs):
			decoder = VideoDecoder(ep)
			ep_len = len(decoder)
			for i in range((window - 1)*frame_skip, ep_len - 1 - frame_skip):
				batch = []
				for j in reversed(range(window)):
					batch.append(i - (j*frame_skip))
				batch.append(i + frame_skip)
				self.frame_samples.append((idx, ep, batch))
		
		"""
		frame_samples looks like this:
		[(0, ep0 Path, [0, 5, 10, 15]),
		 (0, ep0 Path, [1, 6, 11, 16]),
					...				,
		(3, ep3 Path, [5, 10, 15, 20])]
		"""
		
		self.total_len = len(self.frame_samples)
		
	
	def __len__(self):
		return self.total_len

	def __getitem__(self, idx):
		sample = self.frame_samples[idx]
		ep_num, ep_path, frames = sample
		input_frames = frames[:-1]
		target_frames = frames[1:]

		decoder = VideoDecoder(ep_path)
		input_window_tensor = torch.stack([decoder[i] for i in input_frames])
		target_state_tensor = torch.stack([decoder[i] for i in target_frames])
		
		ep_action = self.action_tensor[ep_num]
		action_window = []
		for i in range(len(frames) - 1):
			action = ep_action[frames[i]:frames[i + 1]].sum(dim=0)
			action_window.append(action)

		action_window_tensor = torch.stack(action_window, dim=0)

		return input_window_tensor, action_window_tensor, target_state_tensor

In [ ]:
import torch
from torch import nn
import numpy as np

"""
This is where SIGReg (Sketched Isotropic Gaussian Regularizer) is defined:

This regulizer represents a loss in terms of how well a batch of embeddings 
form an isotropic gaussian and that's just a way of saying how spread apart
the embedding space is. This is a very important term as it helps to prevent
representation collapse of the latent space.
"""

class SIGReg(nn.Module):
    def __init__(self, 
                 knots : int = 17, # the number of points being used to approximate the integral
                 num_proj : int = 1024): # number of random directions the vector will be projected in (M) 
        super().__init__()
        # setting up the trapezoidal approximation of the integral in epps-pulley test
        
        self.num_proj = num_proj
        # this is the tensor split up into points interpolated between 0 to 3 which will be integrated over
        # only need to 0 to 3 as the gaussian essentially goes to 0 after 3 so this serves as a strong approximate
        t = torch.linspace(0, 3, knots, dtype=torch.float32)
        dt = 3 / (knots - 1) 
        
        # these are the weights based on the trapezoid approximation [f(x0) + 2f(x1) + ... 2f(xn-1) + f(xn)]
        weights = torch.full((knots,), 2 * dt, dtype=torch.float32)
        weights[[0, -1]] = dt

        # gaussian characteristic function
        window = torch.exp(-t.square() / 2.0)

        # storing relevant values in buffer as non learnable parameters so that they move across devices with the nn Module
        self.register_buffer("t", t)
        self.register_buffer("phi", window)
        self.register_buffer("weights", weights*window)
    
    def forward(self, Z): # Z : (N, B, D) or (N+1, B, D) if target embeddins is being included
        A = torch.randn(Z.size(-1), self.num_proj, device=Z.device) # tensor of random directions that the embeddings will be projected onto
        A = A.div_(A.norm(p=2, dim=0)) # normalizing the tensor to be unit vectors
        
        # computing the epps-pulley stat
        x_t = (Z @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()
        stat = (err @ self.weights) * Z.size(-2)

        return stat.mean() # returning the mean error across each projection

In [ ]:
from torch import nn
import torch
import math

"""
This is where the individual components of the networks are defined,
more specifically the Attention Heads and MLPs that come together to
make transformers
"""

class TransformerBlock(nn.Module): # pre norm transformer implementation with adaptive layer norm as an option
    def __init__(self, 
                d_model : int = 192,
                nheads : int = 16,
                d_action : int = None,
                dropout : float = 0.1,
                causal_masking : bool = True,
                adaptive : bool = True):
        super().__init__()

        self.d_model = d_model
        self.nheads = nheads
        self.d_action = d_action
        self.hidden_dim = 4 * self.d_model
        self.dropout = dropout
        self.adaptive = adaptive
        
        if self.adaptive:
            self.W_LNparams = nn.Linear(in_features=d_action, out_features=4*d_model) # 2 sets of parameters for each norm
            # initing the weights and bias to 0
            self.W_LNparams.weight.data.zero_()
            self.W_LNparams.bias.data.zero_()
            self.norm = nn.LayerNorm(self.d_model, elementwise_affine=False)
        else:
            self.norm = nn.LayerNorm(self.d_model, elementwise_affine=True)

        self.attn_block = MultiHeadAttention(d_model=d_model,
                                             nheads=nheads,
                                             dropout=self.dropout,
                                             causal_masking=causal_masking)
        self.mlp = nn.Sequential(
            nn.Linear(in_features=self.d_model, out_features=self.hidden_dim),
            nn.GELU(),
            nn.Dropout(self.dropout),
            nn.Linear(in_features=self.hidden_dim, out_features=self.d_model),
            nn.Dropout(self.dropout)
        )                                        

    def forward(self, z, a=None): # positional embeddings should come into this
        # Note: can also add gates to make training more stable if need be
        if self.adaptive:
            params = self.W_LNparams(a)
            scale_one, shift_one, scale_two, shift_two = params.chunk(4, dim=-1) # splitting the tensor into 2 seperate tensors for each param
            residual_one = self.attn_block((1 + scale_one) * self.norm(z) + shift_one)
        else:
            residual_one = self.attn_block(self.norm(z))
        z = z + residual_one
        
        if self.adaptive:
            residual_two = self.mlp((1 + scale_two) * self.norm(z) + shift_two)
        else:
            residual_two = self.mlp(self.norm(z))
        z = z + residual_two

        return z


# shape of the input embeddings: (B, N, d_model)
class MultiHeadAttention(nn.Module): # implemented with causal masking
    def __init__(self, 
                d_model : int = 192, 
                nheads : int = 16,
                dropout : float = 0.1,
                causal_masking : bool = True):

        super().__init__()
        self.d_model = d_model
        self.dropout = dropout
        self.d_k = self.d_model // nheads
        self.H = nheads
        self.causal_masking = causal_masking

        self.drop = nn.Dropout(p=self.dropout)

        # Q V K projection weights
        self.W_q = nn.Linear(in_features=d_model, out_features=d_model)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model)
        
        #output layer
        self.W_o = nn.Linear(in_features=d_model, out_features=d_model)

    def attention(self, Q, K, V):
        # shape of Q K V are (B, H, N, d_k)
        N = Q.shape[2]
        # dot product attention operation
        scores = Q @ K.transpose(-2, -1) # shape: (B, H, N, N)
        scaled_scores =  scores / math.sqrt(self.d_k)

        # temporal masking by adding a mask before softmaxing
        if self.causal_masking:
            mask = torch.tril(torch.ones(N, N, device=Q.device)).bool() 
            masked_scores = scaled_scores.masked_fill(~mask, float("-inf"))
            weights = torch.softmax(masked_scores, dim=-1) # softmaxing to get weights
        else:
            weights = torch.softmax(scaled_scores, dim=-1) # softmaxing to get weights
        weights = self.drop(weights)
        outputs = weights @ V

        return outputs

    def forward(self, X): 
        # shape of X: (B, N, d_model)
        B = X.shape[0]
        N = X.shape[1]

        queries = self.W_q(X).reshape(B, N, self.H, self.d_k).permute(0, 2, 1, 3)
        keys = self.W_k(X).reshape(B, N, self.H, self.d_k).permute(0, 2, 1, 3)
        values = self.W_v(X).reshape(B, N, self.H, self.d_k).permute(0, 2, 1, 3)

        attn_output = self.attention(queries, keys, values)
        attn_output =  attn_output.permute(0, 2, 1, 3).reshape(B, N, self.d_model) # concatenating the output across the different heads

        output = self.W_o(attn_output)
        output = self.drop(output)

        return output

In [ ]:
from torch import nn
import torch

"""
Thought it would be fun to define the architecture of the ViT-Tiny
from scratch so here it is using the transformers I defined as well
"""

class CustomVitTiny(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 npatches : int = 256):
        super().__init__()

        self.npatches = npatches
        self.d_model = d_model

        self.conv_flattener = nn.Conv2d(in_channels=3,
                                        out_channels=d_model,
                                        kernel_size=patch,
                                        stride=patch,
                                        padding=0)
        
        self.transformer_layers = nn.ModuleList([])
        for _ in range(nlayers):
            self.transformer_layers.append(TransformerBlock(d_model=d_model, 
                                           nheads=nheads,                                            
                                           dropout=0.0,
                                           causal_masking=False,
                                           adaptive=False))
        # learned positional embeddings
        self.position_emb = nn.Parameter(torch.randn(npatches + 1, d_model))
        # learned [cls] embedding
        self.cls_emb = nn.Parameter(torch.randn(1, 1, 1, d_model))

    def forward(self, x):
        # shape of x: (B, N, C, H, W)
        B, N, C, H, W = x.shape
        x = x.reshape(B*N, C, H, W)
        emb = self.conv_flattener(x)
        emb = emb.reshape(B, N, self.d_model, self.npatches).permute(0, 1, 3, 2)
        # shape of emb now: (512, 3, 256, 192)
        cls_emb = self.cls_emb.expand(B, N, 1, self.d_model)
        emb = torch.cat([cls_emb, emb], dim=2)
        emb = emb + self.position_emb
        # shape of emb now: (512, 3, 257, 192)
        emb = emb.reshape(B*N, self.npatches + 1, self.d_model) # now ready to go into the transformer

        for layer in (self.transformer_layers):
           emb = layer(emb)
        
        emb = emb.reshape(B, N, self.npatches + 1, self.d_model)
        emb = emb[:, :, 0, :] # shape : (512, 3, 192), this is what the predictor expects as the input

        return emb

In [ ]:
from torch import nn
import torch

"""
This is the network definition for the encoder network which in the paper
was just ViT-Tiny which is a visual transformer with the following parameters:
- patch size of 14, 12 layers, 3 attention heads, and a hidden dimension of 192
- there is a "projection" layer added to the end of this because visual transformers
  typically have layernorms at the end but this ruins the regularization used to
  prevent representation collapse
    - this projection layer is a single layer with batch normalization to undo the
      affects of the layernorm at the end
"""

class EncoderNetwork(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 npatches : int = 256): # npatches is (H/patch_size)*(W/patch_size) 
        super().__init__()
        self.d_model = d_model
        
        self.vit = CustomVitTiny(patch, nheads, nlayers, d_model, npatches)
        self.projector = nn.Linear(in_features=d_model, out_features=d_model)
        self.batch_norm = nn.BatchNorm1d(d_model)
    
    def forward(self, x):
        # shape of input (x): (B, N, C, H, W)
        B, N, C, H, W = x.shape
        z = self.vit(x)
        #projection step
        z = z.reshape(B*N, self.d_model)
        z = self.batch_norm(self.projector(z))
        z = z.reshape(B, N, self.d_model)

        return z

In [ ]:
from torch import nn
import torch


"""
This is the network definition for the predictor network as defined in the original paper:
Transformer Architecture:
- 6 layers, 16 attention heads, 10% dropout
- AdaLN at the beginning of each layer in order to incorporate the actions into the network
- the network takes a window of observation embeddings as the input 
    - for the pushT environment this window is set to 3
"""

class PredictorNetwork(nn.Module):
    def __init__(self,
                 d_model: int = 192,
                 nheads : int = 16,
                 d_action : int = 2,
                 dropout : float = 0.1,
                 nlayers : int = 6,
                 window : int = 3):
        super().__init__()

        self.d_model = d_model
        
        self.transformer_layers = nn.ModuleList([])
        for _ in range(nlayers):
            self.transformer_layers.append(TransformerBlock(d_model, 
                                           nheads, 
                                           d_action, 
                                           dropout,
                                           True,
                                           True))
        # learned positional embeddings
        self.position_emb = nn.Parameter(torch.randn(window, d_model))

        # projection layer
        self.projector = nn.Linear(in_features=d_model, out_features=d_model)
        self.batch_norm = nn.BatchNorm1d(d_model)

    def forward(self, z, a):
        #shape of z: (B, N, D), a: (B, N, D_a)
        B = z.shape[0]
        N = z.shape[1]
        # applying the positional embeddings
        z = z + self.position_emb
        for layer in (self.transformer_layers):
            z = layer(z, a)
        
        z = z.reshape(B*N, self.d_model)
        z = self.batch_norm(self.projector(z))

        z = z.reshape(B, N, self.d_model)

        return z

In [ ]:
import torch
from torch import nn
from pathlib import Path
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from torch.utils.tensorboard import SummaryWriter

import os
import zipfile

def train(device,
          data_dir : Path,
          log_dir : Path,
          save_dir : Path,  
          frame_skip: int = 5,
          lr : float = 3e-4,
          seed : int = 42,
          lambd : float = 0.1,
          epochs : int = 200,
          batch_size : int = 512,
          window_size : int = 3,
          embedding_dim : int = 192,
          action_dim : int = 2,
          pred_nheads : int = 16,
          dropout : float = 0.1,
          pred_nlayers : int = 6,
          patch_size : int = 14,
          enc_nheads : int = 3,
          enc_nlayers : int = 12,
          enc_npatches : int = 256,
          knots : int = 17,
          num_proj : int = 1024):

    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    save_dir.mkdir(parents=True, exist_ok=True)

    # loading and setting up the data 
    train_dataset = PushTDataset(data_dir=data_dir,
                                 frame_skip=frame_skip,
                                 window=window_size,
                                 val=False)

    val_dataset = PushTDataset(data_dir=data_dir,
                               frame_skip=frame_skip,
                               window=window_size, 
                               val=True)

    train_dataloader = DataLoader(dataset=train_dataset, 
                                       batch_size=batch_size, 
                                       shuffle=True)

    val_dataloader = DataLoader(dataset=val_dataset, 
                                     batch_size=batch_size, 
                                     shuffle=False)

    # network definitions
    predictor = PredictorNetwork(d_model=embedding_dim,
                                 nheads=pred_nheads,
                                 d_action=action_dim,
                                 dropout=dropout,
                                 nlayers=pred_nlayers,
                                 window=window_size).to(device)

    encoder = EncoderNetwork(patch=patch_size,
                             nheads=enc_nheads,
                             nlayers=enc_nlayers,
                             d_model=embedding_dim,
                             npatches=enc_npatches).to(device)

    print(f"networks are on device: {device}")

    # defining the losses and the optimizer
    sig_reg = SIGReg(knots=knots,
                     num_proj=num_proj).to(device)
    
    pred_loss_fn = nn.MSELoss()

    optimizer = torch.optim.AdamW(params=list(predictor.parameters()) + list(encoder.parameters()),
                                 lr=lr,
                                 betas=(0.9, 0.95), # these are from the paper
                                 weight_decay=0.05)

    # initializing the logger
    writer = SummaryWriter(log_dir=log_dir)

    # results dict
    results = {"train_loss" : [],
               "val_loss" : []}

    # training loop
    for epoch in tqdm(range(epochs)):
        print(f"Epoch; {epoch}\n------------")

        train_loss = 0 # running loss sum to be averaged after each batch for logging

        encoder.train()
        predictor.train()
        for batch, (X, A, Y) in enumerate(train_dataloader):
            X = X.to(device)
            A = A.to(device)
            Y = Y.to(device)

            embeddings = encoder(X)
            pred_embeddings = predictor(embeddings, A)
            target_embeddings = encoder(Y)

            pred_loss = pred_loss_fn(pred_embeddings, target_embeddings)
            reg_loss = sig_reg.forward(embeddings.transpose(0, 1))

            loss = pred_loss + lambd * reg_loss
            train_loss += loss.item()

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()
        
        train_loss /= len(train_dataloader)

        # validation
        val_loss = 0 # running sum for logging

        encoder.eval()
        predictor.eval()

        with torch.inference_mode():
            for (X, A, Y) in val_dataloader:
                X = X.to(device)
                A = A.to(device)
                Y = Y.to(device)
                
                embeddings = encoder(X)
                pred_embeddings = predictor(embeddings, A)
                target_embeddings = encoder(Y)

                pred_loss = pred_loss_fn(pred_embeddings, target_embeddings)
                reg_loss = sig_reg.forward(embeddings.transpose(0, 1))

                loss = pred_loss + lambd * reg_loss
                val_loss += loss.item()
            
            val_loss /= len(val_dataloader)
        
        results["train_loss"].append(train_loss)
        results["val_loss"].append(val_loss)

        writer.add_scalars(main_tag="Loss",
                           tag_scalar_dict={"train_loss" : train_loss,
                                            "validation_loss" : val_loss},
                           global_step=epoch)

        # saving the models into a single zip after each epoch
        encoder_path = save_dir / "encoder_weights.pth"
        predictor_path = save_dir / "predictor_weights.pth"
        torch.save(encoder.state_dict(), encoder_path)
        torch.save(predictor.state_dict(), predictor_path)

        zip_path = save_dir / f"epoch_{epoch}_model.zip"
        with zipfile.ZipFile(zip_path, 'w') as zipf:
            zipf.write(encoder_path, 'encoder_weights.pth')
            zipf.write(predictor_path, 'predictor_weights.pth')
        
        os.remove(encoder_path)
        os.remove(predictor_path)


        print(f"Training Loss: {train_loss:.4f} | Validation Loss: {val_loss:.4f}")
    
    writer.close()
    
    return results

In [ ]:
PROJECT_DIR = Path("/")

data_dir = PROJECT_DIR / "data"
log_dir = PROJECT_DIR / "logs" 
save_dir = PROJECT_DIR / "models"

device = "cuda" if torch.cuda.is_available() else "cpu"

results = train(device=device,
                data_dir=data_dir,
                log_dir=log_dir,
                save_dir=save_dir)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs